# Comment Category Prediction Challenge

Multi-class text classification (4 classes, macro-F1) for the IIT Madras Machine Learning Practice project.

**Final approach:** TF-IDF on the comment text + numeric/metadata features -> LightGBM with balanced class weights.

## 1. Setup and data loading

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/comment-category-prediction-challenge/Sample.csv
/kaggle/input/comment-category-prediction-challenge/train.csv
/kaggle/input/comment-category-prediction-challenge/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/train.csv")
test = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/Sample.csv")

## 2. Quick exploration

In [4]:
train.shape

(198000, 15)

In [5]:
test.shape

(102000, 14)

In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198000 entries, 0 to 197999
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   created_date  198000 non-null  object
 1   post_id       198000 non-null  int64 
 2   emoticon_1    198000 non-null  int64 
 3   emoticon_2    198000 non-null  int64 
 4   emoticon_3    198000 non-null  int64 
 5   upvote        198000 non-null  int64 
 6   downvote      198000 non-null  int64 
 7   if_1          198000 non-null  int64 
 8   if_2          198000 non-null  int64 
 9   race          52577 non-null   object
 10  religion      52577 non-null   object
 11  gender        52577 non-null   object
 12  disability    198000 non-null  bool  
 13  comment       197999 non-null  object
 14  label         198000 non-null  int64 
dtypes: bool(1), int64(9), object(5)
memory usage: 21.3+ MB


In [8]:
train["label"].value_counts()

label
0    114173
2     62440
1     15918
3      5469
Name: count, dtype: int64

In [9]:
train["race"].value_counts()

race
none      39682
white      5486
black      3869
other      1654
asian      1263
latino      623
Name: count, dtype: int64

In [10]:
train["religion"].value_counts()

religion
none         38249
christian     7191
muslim        4930
jewish        1244
other          547
atheist        219
buddhist       100
hindu           97
Name: count, dtype: int64

In [11]:
train["gender"].value_counts()

gender
none           36161
female          8037
male            7549
transgender      622
other            208
Name: count, dtype: int64

## 3. Final model: TF-IDF + LightGBM

- Text: unigram TF-IDF, 25,000 features, sublinear TF
- Numeric: emoticons, upvote/downvote, if_1/if_2, race/religion/gender/disability flags
- Model: LightGBM multiclass, 500 trees, `class_weight="balanced"` to handle the imbalanced labels
- Trained on the full training set to produce the submission

In [12]:
import pandas as pd
import numpy as np
# from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier
from scipy.sparse import hstack, csr_matrix

train["comment"] = train["comment"].astype(str)
test["comment"] = test["comment"].astype(str)

tfidf = TfidfVectorizer(
    max_features=25000,
    ngram_range=(1,1),
    min_df=3,
    max_df=0.9,
    sublinear_tf=True
)

X_text = tfidf.fit_transform(train["comment"])
X_test_text = tfidf.transform(test["comment"])

num_cols = [
    "emoticon_1","emoticon_2","emoticon_3",
    "upvote","downvote",
    "if_1","if_2",
    "race","religion","gender","disability"
]

for col in num_cols:
    train[col] = pd.to_numeric(train[col], errors="coerce")
    test[col] = pd.to_numeric(test[col], errors="coerce")

train[num_cols] = train[num_cols].fillna(0)
test[num_cols] = test[num_cols].fillna(0)

X_num = csr_matrix(train[num_cols].values.astype(np.float32))
X_test_num = csr_matrix(test[num_cols].values.astype(np.float32))

X = hstack([X_text, X_num])
X_test = hstack([X_test_text, X_test_num])

y = train["label"]

# X_train, X_val, y_train, y_val = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )

lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=4,
    n_estimators=500,
    learning_rate=0.07,
    num_leaves=96,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    force_col_wise=True,
    class_weight="balanced",
    n_jobs=-1
)

# lgbm.fit(X_train, y_train)

# val_pred = lgbm.predict(X_val)
# print("LGBM Macro F1:", f1_score(y_val, val_pred, average="macro"))

lgbm.fit(X, y)

test_pred = lgbm.predict(X_test)

sample_sub ["label"] = test_pred
sample_sub .to_csv("submission.csv", index=False)

[LightGBM] [Info] Total Bins 984951
[LightGBM] [Info] Number of data points in the train set: 198000, number of used features: 17710
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
